# 🧬 1. Conversor FASTA → CSV

Este notebook corresponde a la **etapa inicial de preprocesamiento** del proyecto. Su objetivo es transformar archivos de secuencias proteicas en formato `FASTA` a un formato tabular `CSV`, listo para análisis exploratorio y alimentación de modelos de Machine Learning.

---

## 📋 Descripción General
El pipeline lee archivos `.fasta`, extrae metadatos y secuencias, aplica filtros de longitud y estandarización, y exporta los resultados en `CSV`.

## ⚙️ Funcionalidades Clave
1. **Gestión de rutas:** Configuración de directorios de entrada/salida en Google Drive con creación automática de carpetas.
2. **Dependencias bioinformáticas:** Instalación de `biopython` para parsing robusto de secuencias.
3. **Extracción estructurada:** Por cada registro, se extraen `ID`, `Description`, `Sequence` y `Length`.
4. **Normalización automática:**
   - ✂️ Recorte de secuencias `> MAX_LEN` aminoácidos.
   - 📝 Relleno con `X` para secuencias `< MIN_LEN`.
5. **Procesamiento por lotes:** Escaneo automático con `glob` para convertir todos los `.fasta` del directorio manteniendo la nomenclatura original.

## 📦 Salida Esperada
Archivos `.csv` con el esquema: `id | description | sequence | length`, listos para la fase de **Feature Extraction** y vectorización.

## 📦 Instalación de dependencias

In [1]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 8.1 MB/s eta 0:00:00


## 🔧 Importación de librerías

In [2]:
import os
import glob
import pandas as pd
from Bio import SeqIO
from pathlib import Path

## ⚙️ Configuración de parámetros

In [3]:
INPUT_FASTA = "/kaggle/input/datasets/user/fasta-input"
OUTPUT_CSV = "/kaggle/working/"
os.makedirs(OUTPUT_CSV, exist_ok=True)

MIN_LEN = 10
MAX_LEN = 100
PAD_CHAR = 'X'

## 🔄 Función de conversión FASTA → CSV

In [4]:
def fasta_to_csv(input_fasta: str, output_csv: str) -> pd.DataFrame:
    """
       Convert a FASTA file to CSV with columns:
       id | description | sequence | length
    """
    records = []
    for record in SeqIO.parse(input_fasta, "fasta"):
        records.append({
            "id": record.id,
            "description": record.description,
            "sequence": str(record.seq).upper(),
            "length": len(record.seq)
        })

    df = pd.DataFrame(records)
    if df.empty:
        print(f"   ⚠️ {Path(input_fasta).name} is empty. Omitted.")
        return df

    df_processed = df.copy()

    # Trim sequences > 100
    mask_long = df_processed["length"] > MAX_LEN
    if mask_long.any():
        df_processed.loc[mask_long, "sequence"] = df_processed.loc[mask_long, "sequence"].str[:MAX_LEN]
        print(f"   ✂️ {mask_long.sum()} sequences trimmed to {MAX_LEN} aa")

    # Pad sequences < 10
    mask_short = df_processed["length"] < MIN_LEN
    if mask_short.any():
        df_processed.loc[mask_short, "sequence"] = df_processed.loc[mask_short, "sequence"].str.ljust(MIN_LEN, PAD_CHAR)
        print(f"   📝 {mask_short.sum()} sequences padded to {MIN_LEN} aa with '{PAD_CHAR}'")

    # Recalculate lengths
    df_processed["length"] = df_processed["sequence"].str.len()

    # Save and report
    df_processed.to_csv(output_csv, index=False)
    print(f"✅ {Path(input_fasta).name} → {Path(output_csv).name} "
          f"({len(df_processed)} sequences, max_len={df_processed['length'].max()})")
    return df_processed

## 📂 Procesamiento por lotes

In [5]:
fasta_files = glob.glob(os.path.join(INPUT_FASTA, "*.fasta"))

if not fasta_files:
    print(f"⚠️  No .fasta files were found in: {INPUT_FASTA}")
else:
    print(f"📂 Processing {len(fasta_files)} FASTA file(s)...\n")
    for fasta_path in sorted(fasta_files):
        nombre   = Path(fasta_path).stem
        csv_path = os.path.join(OUTPUT_CSV, f"{nombre}.csv")
        fasta_to_csv(fasta_path, csv_path)
        print("-" * 50)

📂 Processing 6 FASTA file(s)...

✅ test_neg.fasta → test_neg.csv (10771 sequences, max_len=100)
--------------------------------------------------
   📝 267 sequences padded to 10 aa with 'X'
✅ test_pos.fasta → test_pos.csv (4914 sequences, max_len=100)
--------------------------------------------------
   📝 926 sequences padded to 10 aa with 'X'
✅ train_neg.fasta → train_neg.csv (9767 sequences, max_len=100)
--------------------------------------------------
   📝 1054 sequences padded to 10 aa with 'X'
✅ train_pos.fasta → train_pos.csv (9781 sequences, max_len=100)
--------------------------------------------------
   📝 244 sequences padded to 10 aa with 'X'
✅ val_neg.fasta → val_neg.csv (2561 sequences, max_len=100)
--------------------------------------------------
   📝 276 sequences padded to 10 aa with 'X'
✅ val_pos.fasta → val_pos.csv (2564 sequences, max_len=100)
--------------------------------------------------


# 🏁 Resumen del Pipeline FASTA → CSV
Este notebook cierra la fase de **ingesta y estandarización de secuencias**, transformando archivos raw en datasets tabulares validados y listos para consumo en el módulo de *Feature Extraction*.

| 📦 Entregable | 📐 Esquema | 📊 Volumen |
|:---|:---|:---|
| Secuencias tabulares | `id \| description \| sequence \| length` | ✅ 6 archivos CSV (~40.358 secuencias totales) |

### 📌 Logros Clave
- **Normalización de longitud**: Todas las secuencias acotadas al rango `[10, 100]` aminoácidos mediante recorte controlado y relleno con `X`.
- **Preservación de splits**: Se mantuvo la estructura original `train/val/test` × `pos/neg` para garantizar reproducibilidad en validación cruzada.
- **Trazabilidad completa**: Metadatos y longitudes recalculadas exportadas sin pérdida de información estructural.